# Week 3 — Statistics for ML: Descriptive Statistics

**Difficulty:** ⭐⭐ (Beginner-Friendly)  
**Estimated Time:** 2 hours  
**Theme:** House Price Prediction  

---

> **What you will learn:**  
> How to summarize a dataset using a handful of numbers — mean, median, mode, range, and IQR — and why each one matters before you train a single ML model.

## 1. Why Does This Matter for Machine Learning?

Before you build a model you need to **understand your data**. Descriptive statistics are your first tool for that.

Here is why each measure connects directly to ML:

| Statistic | ML Connection |
|-----------|---------------|
| **Mean** | Used in mean-imputation, feature scaling (StandardScaler subtracts the mean) |
| **Median** | Robust imputation when data has outliers |
| **Mode** | Imputing categorical features (most common category) |
| **IQR** | Outlier detection before training; used in RobustScaler |
| **Range** | Tells you whether min-max normalization makes sense |

**Concrete example:** If you train a Linear Regression on house prices without noticing that one $20M mansion is in your dataset, the model's coefficients will be pulled toward that outlier — and your predictions for ordinary homes will be systematically wrong.

## 2. Real-World Analogy: Summarizing a Movie to a Friend

Imagine you just watched a 2-hour film. Your friend asks: *"What was it like?"*

You do not replay the entire movie. Instead you say:
- *"It was mostly a slow-burn thriller (typical pace = median)"*
- *"But there were a few insane action sequences that skewed my impression (outliers affecting the mean)"*
- *"The tone ranged from comedy to horror (range)"*
- *"Most scenes were in the suspense zone (mode / IQR)"*

**Descriptive statistics do exactly this for your dataset.** Instead of handing someone 200 house prices, you hand them 5 numbers and they instantly understand the shape of the data.

A dataset of house prices is our "movie". Let us learn how to summarize it.

## 3. Setup — Creating Synthetic House Price Data

We will create a realistic synthetic dataset: 200 houses in a mid-size city, priced around $300,000 with natural variation, **plus a few mansions added manually** to simulate outliers.

This mirrors real-world datasets where a luxury property sits alongside ordinary homes.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)   # Fix the random seed so results are identical every run

# ── Generate 200 "ordinary" house prices ─────────────────────────────────────
# np.random.normal(mean, std_dev, count)
# mean  = $300,000  (typical city house price)
# std   = $80,000   (houses range roughly $140K–$460K)
# count = 200 houses
base_prices = np.random.normal(loc=300_000, scale=80_000, size=200)

# Clip negative prices — a house cannot cost less than $50,000 in our world
base_prices = np.clip(base_prices, 50_000, None)

# ── Add 5 outlier mansions manually ──────────────────────────────────────────
# These represent luxury properties that exist in every real city dataset.
# They will demonstrate how a few extreme values distort the mean.
mansion_prices = np.array([1_200_000, 1_500_000, 1_800_000, 2_200_000, 3_000_000])

# Combine ordinary homes + mansions into one array
house_prices = np.concatenate([base_prices, mansion_prices])

# ── Wrap in a Pandas DataFrame for easy analysis ─────────────────────────────
df = pd.DataFrame({'price': house_prices})

print(f"Total houses in dataset : {len(df)}")
print(f"Price range             : ${df['price'].min():,.0f}  →  ${df['price'].max():,.0f}")
print(f"First 5 prices          : {df['price'].head().values}")

## 4. The Mean (Average)

### Plain English First

Imagine splitting the total value of all houses equally among every house. Each house would then be worth the **mean** price.

If 10 friends contribute to a dinner bill totaling $500, the **mean** contribution is $50 per person — regardless of who paid more or less.

### Formula

$$\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}$$

- $x_i$ = individual house price  
- $n$ = total number of houses  
- $\bar{x}$ = mean price  

**Step by step:** add up all prices, then divide by the count.

### When the Mean Fails

The mean is sensitive to **outliers**. One $3M mansion pulls the mean upward, even though most houses are around $300K. That is why we also need the median.

In [ ]:
# ── Computing the Mean: Manual Step-by-Step ──────────────────────────────────

# Step 1: Sum all prices
total_price = sum(house_prices)          # Python built-in sum (slow but transparent)
print(f"Step 1 — Sum of all prices   : ${total_price:,.0f}")

# Step 2: Count the number of houses
n_houses = len(house_prices)
print(f"Step 2 — Number of houses    : {n_houses}")

# Step 3: Divide to get the mean
mean_manual = total_price / n_houses
print(f"Step 3 — Mean (manual)       : ${mean_manual:,.0f}")

# ── Cross-check with NumPy (much faster for large arrays) ─────────────────────
mean_numpy = np.mean(house_prices)
print(f"\nMean (numpy)                 : ${mean_numpy:,.0f}")

# ── Cross-check with Pandas ───────────────────────────────────────────────────
mean_pandas = df['price'].mean()
print(f"Mean (pandas)                : ${mean_pandas:,.0f}")

# ── Verify all three match ────────────────────────────────────────────────────
# round() removes floating-point noise before comparison
assert round(mean_manual) == round(mean_numpy) == round(mean_pandas), "Methods disagree!"
print("\nAll three methods agree. Manual calculation confirmed correct.")

## 5. The Median (Middle Value)

### Plain English First

Line up all houses from cheapest to most expensive. The **median** is the price of the house exactly in the middle of that line.

Half the houses cost less than the median. Half cost more. The $3M mansion is at the far end of the line — it does not move the middle value.

### Formula

Sort the data. Then:

- If $n$ is **odd**: median = value at position $\frac{n+1}{2}$
- If $n$ is **even**: median = average of values at positions $\frac{n}{2}$ and $\frac{n}{2}+1$

### Why the Median is "Robust"

"Robust" in statistics means **resistant to outliers**. Even if you add a $100M ultra-mansion to the dataset, the median barely moves — because it only cares about the middle position, not the extreme values.

In [ ]:
# ── Computing the Median: Manual Step-by-Step ─────────────────────────────────

# Step 1: Sort prices from lowest to highest
sorted_prices = np.sort(house_prices)    # Returns a new sorted array

# Step 2: Determine if we have an odd or even number of houses
n = len(sorted_prices)
print(f"Number of houses (n) : {n}")
print(f"n is even            : {n % 2 == 0}")

# Step 3: Find the middle index (Python uses 0-based indexing)
# For n=205, middle indices are 102 and 103 (0-indexed)
if n % 2 == 1:                           # Odd count
    median_manual = sorted_prices[n // 2]
else:                                    # Even count — average the two middle values
    mid = n // 2
    median_manual = (sorted_prices[mid - 1] + sorted_prices[mid]) / 2

print(f"\nMedian (manual)      : ${median_manual:,.0f}")

# ── Cross-check with NumPy ─────────────────────────────────────────────────────
median_numpy = np.median(house_prices)
print(f"Median (numpy)       : ${median_numpy:,.0f}")

# ── The KEY comparison: mean vs median ────────────────────────────────────────
print("\n" + "="*50)
print("IMPACT OF MANSIONS ON MEAN VS MEDIAN")
print("="*50)
print(f"Mean   (with mansions) : ${mean_numpy:,.0f}")
print(f"Median (with mansions) : ${median_numpy:,.0f}")
print(f"Difference             : ${mean_numpy - median_numpy:,.0f}")
print("\nThe mean is pulled ~$20-30K higher by just 5 mansions out of 205 houses.")
print("The median is barely affected — it is robust to outliers.")

## 6. The Mode (Most Frequent Value)

### Plain English First

If you asked 200 people "what is your favorite neighborhood?" and tallied the answers, the most popular answer is the **mode**.

For continuous data like house prices (where every value is slightly different), mode is most useful after **binning** — grouping prices into ranges like "$250K–$300K", "$300K–$350K", etc.

Mode matters most in ML for **categorical features** (e.g., the most common house style, or the most frequent number of bedrooms).

### When Mode is Useful in ML

- Imputing missing values in a categorical column (replace NaN with the most common category)
- Identifying the most common price bracket in a market
- Understanding class imbalance in classification targets

In [ ]:
# ── Mode on Continuous Prices (requires binning first) ────────────────────────

# For truly continuous data, exact mode is rarely meaningful.
# Instead, we bin prices into $50K brackets and find the most common bracket.

# pd.cut divides the price range into equal-width bins
price_bins = pd.cut(
    df['price'],
    bins=range(0, 3_500_000, 50_000),   # Bins: 0–50K, 50K–100K, ..., 3.45M–3.5M
    right=False                          # Intervals are [left, right) — left-inclusive
)

# Count how many houses fall in each bin, sort by count descending
bin_counts = price_bins.value_counts().sort_values(ascending=False)
print("Top 5 most common price brackets:")
print(bin_counts.head())

# The mode bracket is the first entry
modal_bracket = bin_counts.index[0]
print(f"\nModal price bracket: {modal_bracket}")
print("This is the price range where the most houses cluster.")

# ── Mode on Discrete Data (e.g., number of bedrooms) ─────────────────────────
# Simulate a bedrooms column — most houses have 3 bedrooms
np.random.seed(42)
bedrooms = np.random.choice([1, 2, 3, 4, 5], size=205,
                             p=[0.05, 0.20, 0.45, 0.25, 0.05])  # 3BR is most common

# scipy.stats.mode returns the most frequent value and its count
mode_result = stats.mode(bedrooms, keepdims=True)
print(f"\nMost common bedroom count : {mode_result.mode[0]} bedrooms")
print(f"Appears in                : {mode_result.count[0]} out of {len(bedrooms)} houses")
print(f"Pandas mode()             : {pd.Series(bedrooms).mode()[0]} bedrooms")

## 7. Range and IQR (Interquartile Range)

### Plain English First

**Range** is the simplest answer to "how spread out are prices?": just subtract the cheapest from the most expensive.

**Problem:** One $3M mansion makes the range enormous, even though 99% of houses are bunched between $150K–$450K.

**IQR solves this.** It ignores the top 25% and bottom 25% of prices, and measures the spread of the **middle 50%** — the typical houses.

Think of it like this: you are at a school where one student scores 100/100 and another scores 5/100. The range is 95 points — but that does not tell you much about most students. The IQR (middle 50% of scores) tells you where the bulk of the class actually landed.

### Formulas

$$\text{Range} = x_{\max} - x_{\min}$$

$$\text{IQR} = Q_3 - Q_1$$

where:
- $Q_1$ = 25th percentile (value below which 25% of prices fall)
- $Q_3$ = 75th percentile (value below which 75% of prices fall)

In [ ]:
# ── Range: Manual Calculation ────────────────────────────────────────────────
price_min = np.min(house_prices)         # Cheapest house
price_max = np.max(house_prices)         # Most expensive house
price_range = price_max - price_min      # Simple difference

print("RANGE")
print(f"  Minimum price : ${price_min:,.0f}")
print(f"  Maximum price : ${price_max:,.0f}")
print(f"  Range         : ${price_range:,.0f}")
print("  The range is dominated by the $3M mansion — not very informative!")

print()

# ── IQR: Manual Step-by-Step ──────────────────────────────────────────────────
# np.percentile(array, q) returns the value at the q-th percentile
Q1 = np.percentile(house_prices, 25)     # 25th percentile = 1st quartile
Q3 = np.percentile(house_prices, 75)     # 75th percentile = 3rd quartile
IQR = Q3 - Q1                            # Spread of the middle 50% of prices

print("IQR (INTERQUARTILE RANGE)")
print(f"  Q1  (25th percentile) : ${Q1:,.0f}")
print(f"  Q3  (75th percentile) : ${Q3:,.0f}")
print(f"  IQR = Q3 - Q1         : ${IQR:,.0f}")
print(f"  The middle 50% of houses are priced within a ${IQR:,.0f} window.")

# ── Cross-check with scipy ─────────────────────────────────────────────────────
from scipy.stats import iqr as scipy_iqr
print(f"\n  IQR (scipy)           : ${scipy_iqr(house_prices):,.0f}")
print("\nNotice: IQR ($~108K) is MUCH smaller than Range ($~2.9M).")
print("IQR tells you the spread of typical houses; Range is polluted by mansions.")

## 8. Outlier Detection Using the 1.5 × IQR Rule

### Plain English First

The **1.5 × IQR rule** (also called Tukey's fence) gives us a principled way to flag unusual prices:

- Any house priced **below** $Q_1 - 1.5 \times IQR$ is suspiciously cheap (possible data error or fire sale)
- Any house priced **above** $Q_3 + 1.5 \times IQR$ is suspiciously expensive (luxury outlier)

This is the exact rule that **box plots use** to draw their whiskers and plot individual outlier dots.

### Formula

$$\text{Lower fence} = Q_1 - 1.5 \times IQR$$
$$\text{Upper fence} = Q_3 + 1.5 \times IQR$$

Values outside these fences are flagged as outliers.

In [ ]:
# ── Tukey's Fence: Outlier Detection ─────────────────────────────────────────

# Compute the fences using Q1 and IQR from the previous cell
lower_fence = Q1 - 1.5 * IQR            # Any price below this is a low outlier
upper_fence = Q3 + 1.5 * IQR            # Any price above this is a high outlier

print("OUTLIER DETECTION: TUKEY'S 1.5 x IQR RULE")
print(f"  Q1            : ${Q1:,.0f}")
print(f"  Q3            : ${Q3:,.0f}")
print(f"  IQR           : ${IQR:,.0f}")
print(f"  Lower fence   : ${lower_fence:,.0f}  (Q1 - 1.5*IQR)")
print(f"  Upper fence   : ${upper_fence:,.0f}  (Q3 + 1.5*IQR)")

# Identify which houses fall outside the fences
# A boolean mask: True for every price that IS an outlier
is_low_outlier  = house_prices < lower_fence
is_high_outlier = house_prices > upper_fence
is_outlier      = is_low_outlier | is_high_outlier   # '|' = logical OR

n_outliers = np.sum(is_outlier)          # Count of True values
print(f"\n  Low  outliers  : {np.sum(is_low_outlier)} houses")
print(f"  High outliers  : {np.sum(is_high_outlier)} houses")
print(f"  Total outliers : {n_outliers} out of {len(house_prices)} houses")

# Show the actual outlier prices, sorted
outlier_prices = np.sort(house_prices[is_outlier])
print(f"\n  Outlier prices : {['${:,.0f}'.format(p) for p in outlier_prices]}")
print("\n  We can see our 5 manually-added mansions were correctly flagged!")

# ── Effect of removing outliers on the mean ───────────────────────────────────
clean_prices = house_prices[~is_outlier]  # '~' inverts the mask (keep non-outliers)
print(f"\n  Mean WITH  outliers : ${np.mean(house_prices):,.0f}")
print(f"  Mean WITHOUT outliers: ${np.mean(clean_prices):,.0f}")
print(f"  Median (unchanged)  : ${np.median(house_prices):,.0f}")
print("\n  Removing outliers brings the mean close to the median.")

## 9. Putting It All Together — Complete Summary Statistics

In [ ]:
# ── Full Summary Statistics Table ─────────────────────────────────────────────
# pandas .describe() gives us a lot at once — but let us build our own
# summary to understand every piece.

summary = {
    'Count'         : len(house_prices),
    'Mean'          : np.mean(house_prices),
    'Median'        : np.median(house_prices),
    'Std Dev'       : np.std(house_prices, ddof=0),   # Population std (ddof=0)
    'Min'           : np.min(house_prices),
    'Q1 (25%)'      : np.percentile(house_prices, 25),
    'Q3 (75%)'      : np.percentile(house_prices, 75),
    'Max'           : np.max(house_prices),
    'Range'         : np.max(house_prices) - np.min(house_prices),
    'IQR'           : np.percentile(house_prices, 75) - np.percentile(house_prices, 25),
    'Outliers (IQR)': int(np.sum(is_outlier)),
}

print("COMPLETE DESCRIPTIVE SUMMARY — House Prices ($)")
print("-" * 45)
for stat, value in summary.items():
    if isinstance(value, float):
        print(f"  {stat:<20} : ${value:>12,.0f}")
    else:
        print(f"  {stat:<20} : {value:>13}")

print("\n--- pandas .describe() (cross-check) ---")
print(df['price'].describe().apply(lambda x: f"${x:,.0f}"))

## 10. Visualizations — Histogram and Box Plot

Two charts that every data scientist uses to understand a distribution at a glance:

- **Histogram:** shows how many houses fall in each price bracket (the "shape" of the distribution)
- **Box plot:** shows Q1, median, Q3, whiskers (fences), and individual outlier dots — all the stats from this notebook in one chart

In [ ]:
# ── Figure Setup: Two charts side by side ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))    # 1 row, 2 columns
fig.suptitle('House Price Distribution — Descriptive Statistics',
             fontsize=15, fontweight='bold', y=1.02)

# ── LEFT CHART: Histogram ─────────────────────────────────────────────────────
ax1 = axes[0]
ax1.hist(house_prices, bins=40, color='steelblue', edgecolor='white', alpha=0.85)

# Add vertical lines for mean and median so we can see the skew effect
ax1.axvline(np.mean(house_prices),   color='crimson',  linewidth=2.0,
            linestyle='--', label=f'Mean   = ${np.mean(house_prices):,.0f}')
ax1.axvline(np.median(house_prices), color='darkorange', linewidth=2.0,
            linestyle='-',  label=f'Median = ${np.median(house_prices):,.0f}')

ax1.set_title('Histogram: Price Distribution', fontsize=12)
ax1.set_xlabel('House Price ($)', fontsize=11)
ax1.set_ylabel('Number of Houses', fontsize=11)
ax1.legend(fontsize=10)
# Format x-axis labels in $K or $M
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
ax1.tick_params(axis='x', rotation=30)

# ── RIGHT CHART: Box Plot ─────────────────────────────────────────────────────
ax2 = axes[1]
# sns.boxplot draws: box (Q1–Q3), line (median), whiskers (fences), dots (outliers)
sns.boxplot(y=house_prices, ax=ax2, color='lightsteelblue',
            flierprops=dict(marker='o', color='crimson', markersize=5))

# Annotate the key statistics on the box plot
annotation_config = dict(fontsize=9, color='navy')
ax2.annotate(f'Q3 = ${Q3:,.0f}',  xy=(0.5, Q3),  xytext=(0.65, Q3),  **annotation_config)
ax2.annotate(f'Median',            xy=(0.5, median_numpy), xytext=(0.65, median_numpy), **annotation_config)
ax2.annotate(f'Q1 = ${Q1:,.0f}',  xy=(0.5, Q1),  xytext=(0.65, Q1),  **annotation_config)

ax2.set_title('Box Plot: All Stats at Once', fontsize=12)
ax2.set_ylabel('House Price ($)', fontsize=11)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y/1e6:.1f}M' if y >= 1e6 else f'${y/1e3:.0f}K'))

plt.tight_layout()
plt.show()

print("Observations:")
print("  Histogram: The tall bar near $300K = the modal price bracket.")
print("             The mean (red dashed) is pulled right by mansions.")
print("             The median (orange) stays near the visual centre.")
print("  Box plot : Red dots above the upper whisker = our 5 mansions.")
print("             The box shows Q1–Q3 (the IQR window).")

## 11. Demonstrating Outlier Effect: Mean vs Median

In [ ]:
# ── Progressive Outlier Injection: Watch the Mean Shift ──────────────────────
# We will add mansions one-by-one and track how mean and median respond.
# This demonstrates visually why the median is "robust".

# Start with only the 200 base (ordinary) house prices
base_only = base_prices.copy()

means   = []    # Mean at each step
medians = []    # Median at each step
labels  = []    # X-axis labels

# Step 0: baseline (no mansions)
current = base_only.copy()
means.append(np.mean(current))
medians.append(np.median(current))
labels.append('0 mansions')

# Add each mansion one at a time and record the updated statistics
for i, mansion in enumerate(mansion_prices, start=1):
    current = np.append(current, mansion)   # Add one mansion to the dataset
    means.append(np.mean(current))
    medians.append(np.median(current))
    labels.append(f'+${mansion/1e6:.1f}M mansion')

# ── Plot the progression ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

x_pos = range(len(labels))    # X-axis positions (0, 1, 2, 3, 4, 5)

ax.plot(x_pos, means,   'o-', color='crimson',    linewidth=2, markersize=7,
        label='Mean (sensitive to outliers)')
ax.plot(x_pos, medians, 's-', color='steelblue',  linewidth=2, markersize=7,
        label='Median (robust to outliers)')

# Draw a horizontal reference line at the baseline mean
ax.axhline(means[0], color='crimson', linewidth=0.8, linestyle=':', alpha=0.5)

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Price ($)', fontsize=11)
ax.set_title('Mean vs Median as Mansions Are Added One by One', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y/1e3:.0f}K'))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("KEY INSIGHT:")
print(f"  Mean  shifted by: ${means[-1] - means[0]:,.0f} after adding 5 mansions")
print(f"  Median shifted by: ${medians[-1] - medians[0]:,.0f} after adding 5 mansions")
print("  The mean is very sensitive; the median barely moved.")

## 12. Quick Reference — When to Use Which Statistic

| Situation | Use This |
|-----------|----------|
| Symmetric data, no outliers | **Mean** |
| Skewed data or outliers present | **Median** |
| Categorical data or discrete counts | **Mode** |
| Overall spread (context allows outliers) | **Range** |
| Robust spread / outlier detection | **IQR** |
| ML feature scaling without outlier influence | **Median + IQR** (RobustScaler) |
| ML feature scaling on clean data | **Mean + Std** (StandardScaler) |

**scikit-learn's `RobustScaler`** scales features using the median and IQR — exactly because these are robust to outliers. Now you know why it was designed that way.

## 13. Self-Check Questions

Answer these before looking at the solutions below. They test conceptual understanding, not just code.

---

**Question 1:** A neighborhood has houses priced at $200K, $210K, $220K, $230K, and $2,000,000. Which measure best represents a "typical" house price, and why?

**Question 2:** What does IQR tell you that range does not?

**Question 3:** If mean > median in a price distribution, what does that suggest about the shape of the distribution?

**Question 4:** You are imputing missing values in a house price column that contains some luxury outliers. Should you use mean or median imputation? Why?

**Question 5:** The IQR rule flags a $85K house as a low outlier. Should you automatically remove it from your dataset?

## 13. Answers

---

**Answer 1:**  
The **median** ($220K) best represents a typical house. The mean is pulled to approximately $572K by the $2M mansion — far above any of the four ordinary houses. The median sits at the actual middle of the sorted list and is unaffected by the single extreme value. In real estate reporting, this is why agencies report median home prices rather than means.

---

**Answer 2:**  
Range only tells you the distance between the single cheapest and single most expensive house — it is dominated by extreme values. IQR ignores the top 25% and bottom 25% and measures the spread of the **middle 50%** of houses. IQR tells you how prices vary for typical properties, which is far more useful for understanding the market and for outlier detection.

---

**Answer 3:**  
When mean > median, the distribution is **right-skewed** (positively skewed). There is a long right tail — a small number of very high-priced homes pulling the mean upward. The bulk of the houses sit below the mean. This is the typical shape of real-world house price distributions (luxury properties create the right tail).

---

**Answer 4:**  
Use **median imputation**. Luxury outliers inflate the mean, so filling in missing values with the mean would artificially raise the imputed prices and introduce bias into your model. The median is unaffected by those outliers and represents a more realistic "typical" price for the missing values.

---

**Answer 5:**  
Not automatically. A statistical flag is a prompt to investigate, not an automatic deletion order. The $85K house could be a legitimate foreclosure sale, a rural property, or a data entry error. Investigate the source: if it is a real transaction, keep it. If it is a data error (e.g., $850K entered as $85K), correct or remove it. Blindly deleting flagged values can remove important real-world signal from your model.

In [ ]:
# ── Final Recap: All Key Statistics in One Place ──────────────────────────────
print("NOTEBOOK 1 — DESCRIPTIVE STATISTICS: FINAL SUMMARY")
print("=" * 55)
print(f"  Dataset         : {len(house_prices)} house prices (200 normal + 5 mansions)")
print()
print(f"  Mean            : ${np.mean(house_prices):>12,.0f}  ← pulled by outliers")
print(f"  Median          : ${np.median(house_prices):>12,.0f}  ← robust to outliers")
print(f"  Modal bracket   : {modal_bracket}")
print()
print(f"  Min             : ${np.min(house_prices):>12,.0f}")
print(f"  Q1              : ${np.percentile(house_prices,25):>12,.0f}")
print(f"  Q3              : ${np.percentile(house_prices,75):>12,.0f}")
print(f"  Max             : ${np.max(house_prices):>12,.0f}")
print()
print(f"  Range           : ${price_range:>12,.0f}  ← inflated by mansions")
print(f"  IQR             : ${IQR:>12,.0f}  ← robust spread measure")
print()
print(f"  Lower fence     : ${lower_fence:>12,.0f}")
print(f"  Upper fence     : ${upper_fence:>12,.0f}")
print(f"  Outliers found  : {int(np.sum(is_outlier))} houses")
print()
print("Next notebook: Notebook 02 — Measures of Spread")
print("  Variance, Standard Deviation, Skewness, Kurtosis")